In [ ]:
import json
import random

import polars as pl

from glob import glob
from pathlib import Path
from datetime import datetime
from rich.pretty import pprint

from common.utils.json_data import load_json

In [ ]:
def get_state(file_path: str) -> dict | None:
    try:
        state = load_json(file_path)
        mod_time = datetime.fromtimestamp(Path(file_path).stat().st_mtime)
        state = state | {  # type: ignore
            "mod_date": mod_time,
        }

        return state
    except json.JSONDecodeError:
        print(f"ERROR: {file_path}")
        return None


In [ ]:
file_paths = glob("/resources/states/*.json")
print(f"file_paths: {len(file_paths)}")

In [ ]:
states = (get_state(fp) for fp in file_paths)
states = [s for s in states if s is not None]

print(f"states: {len(states)}")

In [ ]:
start_date = datetime(2026, 3, 12)
end_date = datetime(2026, 3, 18)

filtered_states = [
    s
    for s in states
    if start_date <= s["mod_date"] <= end_date and s["recorder_ok"]  # type: ignore
]

print(f"filtered_states: {len(filtered_states)}")

In [ ]:
state = random.choice(filtered_states)  # type: ignore
pprint(state)

In [ ]:
accepetd_states = [s for s in filtered_states if s["message_accepted"]]  # type:ignore
pprint(f"accepetd_states: {len(accepetd_states)}")

In [ ]:
rejected_states = [s for s in filtered_states if not s["message_accepted"]]  # type:ignore
pprint(f"rejected_states: {len(rejected_states)}")

In [ ]:
rejected_df = pl.DataFrame(rejected_states)
rejected_df = rejected_df.select(
    [
        "audio_transcription",
        "rejection_reason",
        "detected_language",
    ]
)

rejected_df

In [ ]:
rejected_df.write_csv("/resources/rejected-queries.csv")